**GitHub-context multi-scale recovery.** This figure extends the replay-token study to pinned public repository snapshots and three task scales. The task-specific code is kept in separate recovery directories; the metric charges only post-policy autonomous recovery tokens. All rows use the same `deepseek-v4-flash` model and `strace` trace boundary.

In [1]:
# ipython -c "%run plot_github_token_tasks.ipynb"

import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8
def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

STYLES = {
    'causal': dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2),
    'temporal_checkpoint': dict(color='#e78129', marker='x', linestyle=':', linewidth=0.9, markersize=3.6, markeredgewidth=0.9),
    'whole_branch_abort': dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none'),
}
LABELS = {
    'causal': 'AgentTX causal (ours)',
    'temporal_checkpoint': 'optimistic checkpoint',
    'whole_branch_abort': 'whole-branch abort',
}

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

def provider_result(name):
    provider = os.environ.get('AGENTTX_PROVIDER', 'deepseek')
    preferred = RESULTS / provider / name
    if preferred.exists():
        return preferred
    return RESULTS / name

df = pd.read_csv(provider_result('github_token_tasks_raw.csv'))
for column in ['total_tokens', 'success', 'independent_retained', 'tests_rc']:
    if column in ['success', 'independent_retained']:
        df[column] = df[column].astype(str).str.lower().eq('true')
    else:
        df[column] = pd.to_numeric(df[column], errors='coerce')
df['scale'] = pd.Categorical(df['scale'], categories=['short', 'medium', 'long'], ordered=True)
df = df.sort_values(['scale', 'mode', 'repeat'])
summary = (df.groupby(['scale', 'mode'], observed=False, as_index=False)
             .agg(total_tokens=('total_tokens', 'mean'), success=('success', 'mean'),
                  independent_retained=('independent_retained', 'mean')))

x = np.arange(3)
scale_labels = ['short', 'medium', 'long']
fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
handles = []
ax = plt.subplot(1, 2, 1)
for mode in STYLES:
    rows = summary[summary['mode'] == mode].set_index('scale').reindex(scale_labels)
    handle, = ax.plot(x, rows['total_tokens'].to_numpy(dtype=float), **STYLES[mode], label=LABELS[mode])
    handles.append(handle)
ax.set_xticks(x, scale_labels)
ax.set_ylabel('Post-policy recovery tokens', fontsize=8)
ax.set_xlabel('(a) GitHub-context task scale', fontsize=7)
ax.tick_params(axis='both', labelsize=7)
ax.set_ylim(bottom=0)

ax = plt.subplot(1, 2, 2)
width = 0.12
offsets = {'causal': -width, 'temporal_checkpoint': 0.0, 'whole_branch_abort': width}
for metric, label, hatch in [('success', 'task success', ''), ('independent_retained', 'independent work retained', '//')]:
    for mode in STYLES:
        rows = summary[summary['mode'] == mode].set_index('scale').reindex(scale_labels)
        xpos = x + offsets[mode] + (0.0 if metric == 'success' else 0.38)
        ax.bar(xpos, rows[metric].to_numpy(dtype=float), width=width,
               color=STYLES[mode]['color'], alpha=0.72 if metric == 'success' else 0.38,
               edgecolor='black', linewidth=0.35, hatch=hatch,
               label=f'{LABELS[mode]}: {label}')
ax.set_xticks(x + 0.19, scale_labels)
ax.set_ylabel('Rate', fontsize=8)
ax.set_xlabel('(b) Correctness outcomes', fontsize=7)
ax.set_ylim(0, 1.08)
ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.06), ncol=3,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-GitHub-Token-Tasks.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-GitHub-Token-Tasks.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

display(summary[['scale', 'mode', 'total_tokens', 'success', 'independent_retained']])
assert set(df['model']) == {'deepseek-v4-flash'}
assert set(df['trace_backend']) == {'strace'}


,scale,mode,total_tokens,success,independent_retained
0,short,causal,47977.0,1.0,1.0
1,short,temporal_checkpoint,28209.0,1.0,1.0
2,short,whole_branch_abort,100939.0,0.0,0.0
3,medium,causal,41393.0,1.0,1.0
4,medium,temporal_checkpoint,61313.0,0.0,1.0
5,medium,whole_branch_abort,68449.0,0.0,0.0
6,long,causal,41183.0,1.0,1.0
7,long,temporal_checkpoint,69643.0,0.0,1.0
8,long,whole_branch_abort,65505.0,0.0,0.0
